# Actividad: Backpropagation y Funciones de Activación
Implementación de perceptrón, red neuronal de una capa oculta y red multicapa para clasificación binaria.
Se evidencia el proceso de aprendizaje mediante **feedforward, cálculo de pérdida y backpropagation**.

## 1. Importar librerías

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

## 2. Crear dataset de clasificación binaria

In [ ]:
X, y = make_classification(n_samples=500, n_features=2, n_redundant=0,
                           n_clusters_per_class=1, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

## 3. Funciones de activación

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(x):
    return x * (1 - x)

def relu(x):
    return np.maximum(0, x)

def relu_derivative(x):
    return np.where(x > 0, 1, 0)

## 4. Modelo 1: Perceptrón

In [ ]:
class Perceptron:

    def __init__(self, num_features, lr=0.01, epochs=1000):
        self.weights = np.zeros(num_features + 1)
        self.lr = lr
        self.epochs = epochs

    def activation(self, x):
        return 1 if x >= 0 else 0

    def predict(self, x):
        z = np.dot(x, self.weights[1:]) + self.weights[0]
        return self.activation(z)

    def train(self, X, y):
        for _ in range(self.epochs):
            for xi, target in zip(X, y):
                update = self.lr * (target - self.predict(xi))
                self.weights[1:] += update * xi
                self.weights[0] += update

## 5. Modelo 2: Red neuronal de una capa oculta

In [ ]:
class NeuralNetwork:

    def __init__(self, input_size, hidden_size, output_size, lr=0.01):
        self.lr = lr

        self.W1 = np.random.randn(input_size, hidden_size)
        self.b1 = np.zeros((1, hidden_size))

        self.W2 = np.random.randn(hidden_size, output_size)
        self.b2 = np.zeros((1, output_size))

    def forward(self, X):
        self.z1 = np.dot(X, self.W1) + self.b1
        self.a1 = sigmoid(self.z1)

        self.z2 = np.dot(self.a1, self.W2) + self.b2
        self.a2 = sigmoid(self.z2)

        return self.a2

    def backward(self, X, y, output):

        error = y.reshape(-1,1) - output

        d_output = error * sigmoid_derivative(output)

        error_hidden = d_output.dot(self.W2.T)
        d_hidden = error_hidden * sigmoid_derivative(self.a1)

        self.W2 += self.a1.T.dot(d_output) * self.lr
        self.b2 += np.sum(d_output, axis=0, keepdims=True) * self.lr

        self.W1 += X.T.dot(d_hidden) * self.lr
        self.b1 += np.sum(d_hidden, axis=0, keepdims=True) * self.lr

## 6. Entrenamiento del modelo

In [ ]:
nn = NeuralNetwork(2,5,1,0.01)

loss_history = []

for epoch in range(1000):

    output = nn.forward(X_train)

    loss = np.mean((y_train.reshape(-1,1) - output)**2)
    loss_history.append(loss)

    nn.backward(X_train, y_train, output)

print("Entrenamiento finalizado")

## 7. Evaluación del modelo

In [ ]:
pred = nn.forward(X_test)
pred_class = (pred > 0.5).astype(int)

accuracy = np.mean(pred_class.flatten() == y_test)

print("Accuracy:", accuracy)

## 8. Visualizar el aprendizaje

In [ ]:
plt.plot(loss_history)
plt.title("Evolución del error durante el entrenamiento")
plt.xlabel("Épocas")
plt.ylabel("Loss")
plt.show()